## Prerequisites

Before starting, ensure you have:
- [ ] AWS Account with appropriate permissions
- [ ] IAM User with CloudFormation, EC2, RDS, S3, ECS, Route53 permissions. Alternatively, you may assign the IAM user the AdministratorAccess policy. This ensures full authentication and access to all AWS services required by the system.
- [ ] Git installed on your machine
- [ ] Terminal/Command Line access

## Stack Deployment Order

The following stacks must be deployed in this exact order:
1. **Networking** - VPC, Subnets, Security Groups
2. **S3** - Storage buckets
3. **LoadBalancer** - Application Load Balancer and Target Groups
4. **RDS** - Database infrastructure
5. **Route53** - DNS routing
6. **CloudWatch** - Monitoring and logging
7. **SQS** - Message queues
8. **ECR** - Container registries
9. **OpenSearch** - Search and analytics
10. **SecretsManager** - Database and OpenSearch credentials (depends on OpenSearch)
11. **ECS** - Container services (depends on all above)
12. **SES** - Email service 
13. **Lambda** - Serverless functions 
14. **EventBridge** - Event scheduling 

## 1. Clone the Repository

In [ ]:
%%bash
git clone https://github.com/bmir-datahub/datahub-cloud-replication.git
cd datahub-cloud-replication
ls -la  # Verify files are present

## 2. Install AWS CLI

In [ ]:
%%bash
# For MacOS:
brew install awscli
# Verify installation
aws --version

## 3. Configure AWS Credentials

**Steps to get AWS credentials:**
1. Log into AWS Console
2. Go to IAM → Users → Select your user
3. Security Credentials tab → Create Access Key
4. Download and save the credentials securely

Replace `YOUR_ACCESS_KEY_ID` and `YOUR_SECRET_ACCESS_KEY` with actual values:

In [ ]:
%%bash
mkdir -p ~/.aws
cat <<EOL >> ~/.aws/credentials
[datahub-rep]
aws_access_key_id=YOUR_ACCESS_KEY_ID
aws_secret_access_key=YOUR_SECRET_ACCESS_KEY
EOL

## 4. Set Environment and Verify AWS Configuration
*Set environment once [dev, test, prod] - all deployment cells will use these values*

In [11]:
import os

# Set environment variables (change 'dev' to 'test' or 'prod' as needed)
ENV = input("Enter environment (dev, test, prod): ").strip()
if ENV not in ['dev', 'test', 'prod']:
    raise ValueError("Invalid environment. Choose from 'dev', 'test', or 'prod'.")

# IMPORTANT: Unset any existing AWS credentials from environment
%env AWS_ACCESS_KEY_ID=
%env AWS_SECRET_ACCESS_KEY=
%env AWS_SESSION_TOKEN=

# Now set the profile and region
os.environ['AWS_PROFILE'] = 'datahub-rep'
os.environ['AWS_DEFAULT_REGION'] = 'us-east-1'

print(f'Environment: {ENV}')
print('Testing AWS connectivity...')

# !aws sts get-caller-identity
!aws sts get-caller-identity 
print('✅ AWS configuration verified')

env: AWS_ACCESS_KEY_ID=
env: AWS_SECRET_ACCESS_KEY=
env: AWS_SESSION_TOKEN=
Environment: dev
Testing AWS connectivity...
{
    "UserId": "AIDA3HX6WPYUFZECRFJLX",
    "Account": "772554522152",
    "Arn": "arn:aws:iam::772554522152:user/datahub-dev"
}
✅ AWS configuration verified


In [10]:
AWS_ACCOUNT_ID = !aws sts get-caller-identity --query Account --output text
AWS_ACCOUNT_ID = AWS_ACCOUNT_ID[0]  # Extract string from list

REGION = "us-east-1"
REPO = "datahub-user-service/dev"

# Build the ECR URI
ECR_URI = f"{AWS_ACCOUNT_ID}.dkr.ecr.{REGION}.amazonaws.com/{REPO}:latest"

print(f"ECR URI: {ECR_URI}")
print()

# Check the manifest in ECR
!docker manifest inspect {ECR_URI}

ECR URI: 772554522152.dkr.ecr.us-east-1.amazonaws.com/datahub-user-service/dev:latest

{
   "schemaVersion": 2,
   "mediaType": "application/vnd.oci.image.index.v1+json",
   "manifests": [
      {
         "mediaType": "application/vnd.oci.image.manifest.v1+json",
         "size": 1631,
         "digest": "sha256:0c061f5e30b9019b760e1856e778a44126c7a8d72586a6338d82030de2da30d8",
         "platform": {
            "architecture": "amd64",
            "os": "linux"
         }
      },
      {
         "mediaType": "application/vnd.oci.image.manifest.v1+json",
         "size": 566,
         "digest": "sha256:dd8ffe87717a29682d77a9c4ec7b4cd0a2bde995bd15bf5fd0b30dcdad5d00b5",
         "platform": {
            "architecture": "unknown",
            "os": "unknown"
         }
      }
   ]
}


### Optional: Assign the IAM user the AdministratorAccess policy

In [18]:
!aws iam attach-user-policy \
  --user-name datahub-dev \
  --policy-arn arn:aws:iam::aws:policy/AdministratorAccess

---
# 🚀 Stack Deployment

**Important**: Wait for each stack to complete before proceeding to the next one. Check the AWS Console CloudFormation page to monitor progress.

## 5. Deploy Networking Stack
*Creates VPC, subnets, security groups, and networking infrastructure*

In [18]:
print('🔗 Deploying Networking Stack...')

!aws cloudformation deploy \
  --stack-name DataHub-Networking-{ENV} \
  --template-file modules/Networking.yaml \
  --parameter-overrides file://parameters-{ENV}.json \
  --capabilities CAPABILITY_NAMED_IAM \
  --region us-east-1 \
  --tags projectname=datahub environment={ENV}

print('✅ Networking Stack deployment complete')

🔗 Deploying Networking Stack...

Waiting for changeset to be created..
Waiting for stack create/update to complete
Successfully created/updated stack - DataHub-Networking-dev
✅ Networking Stack deployment complete


## 6. Deploy S3 Stack
*Creates S3 buckets for application data, uploads, and artifacts*

### ⚠️ Important: Configure DataHubUniqueId

Before deploying the S3 stack, you **must** update the `DataHubUniqueId` parameter in your `parameters-{ENV}.json` file to ensure S3 bucket names are globally unique.

**Why is this required?**
- S3 bucket names must be globally unique across all AWS accounts
- The default value `stanford` may already be taken
- Choose a unique identifier for your organization (e.g., your institution name, project code, or random string)

**Steps:**

1. Open `parameters-{ENV}.json` (e.g., `parameters-dev.json`)
2. Update the `DataHubUniqueId` value:
3. Use lowercase letters, numbers, and hyphens only (no underscores or spaces)
4. Keep it short (5-20 characters recommended)

**Bucket names that will be created:**
- `datahub-upload-portal-{DataHubUniqueId}-{ENV}`
- `sftp-datahub-{DataHubUniqueId}-{ENV}`
- `datahub-review-{DataHubUniqueId}-{ENV}`
- `datahub-default-{DataHubUniqueId}-{ENV}`
- `datahub-lambda-artifacts-{DataHubUniqueId}-{ENV}`

If deployment fails with "BucketAlreadyExists" error, choose a different `DataHubUniqueId` value.

In [12]:
print('🗄️ Deploying S3 Stack...')

!aws cloudformation deploy \
  --stack-name DataHub-S3-{ENV} \
  --template-file modules/S3.yaml \
  --parameter-overrides file://parameters-{ENV}.json \
  --capabilities CAPABILITY_NAMED_IAM \
  --region us-east-1 \
  --tags projectname=datahub environment={ENV}

print('✅ S3 Stack deployment complete')

🗄️ Deploying S3 Stack...

Waiting for changeset to be created..
Waiting for stack create/update to complete
Successfully created/updated stack - DataHub-S3-dev
✅ S3 Stack deployment complete


## 7. Deploy Load Balancer Stack
*Creates Application Load Balancer, target groups, and routing rules*

In [15]:
print('⚖️ Deploying Load Balancer Stack...')

!aws cloudformation deploy \
  --stack-name DataHub-LoadBalancer-{ENV} \
  --template-file modules/LoadBalancer.yaml \
  --parameter-overrides file://parameters-{ENV}.json \
  --capabilities CAPABILITY_NAMED_IAM \
  --region us-east-1 \
  --tags projectname=datahub environment={ENV}

print('✅ Load Balancer Stack deployment complete')

⚖️ Deploying Load Balancer Stack...

Waiting for changeset to be created..
Waiting for stack create/update to complete
Successfully created/updated stack - DataHub-LoadBalancer-dev
✅ Load Balancer Stack deployment complete


## 8. Deploy RDS Stack
*Creates RDS database instances and related infrastructure*

### ⚠️ Important: Configure RDS Security Group Before Deployment

**Before deploying the RDS Stack**, you need to add your local public IP address to the RDS security group configuration in `RDS.yaml`. This will allow you to connect to the RDS instance from your local machine for database schema deployment.

#### Steps:

1. **Get your public IP address:**
   ```bash
   curl -4 ifconfig.me
   ```

2. **Update `modules/RDS.yaml` at line 153:**
   - Open `modules/RDS.yaml` in your editor
   - Find line 153 which contains: `- CidrIp: "174.160.114.86/32"`
   - Replace `"174.160.114.86/32"` with your public IP address followed by `/32`
   - Example: If your IP is `203.0.113.42`, change it to `"203.0.113.42/32"`
   - Optionally update the `Description` field on line 154 to identify your workstation

**Note:** The `/32` suffix means only your specific IP address will be allowed. If your IP changes (e.g., when connecting from a different network), you'll need to update this value again.

In [3]:
print('🗃️ Deploying RDS Stack... (This may take 10-15 minutes)')

!aws cloudformation deploy \
  --stack-name DataHub-RDS-{ENV} \
  --template-file modules/RDS.yaml \
  --parameter-overrides file://parameters-{ENV}.json \
  --capabilities CAPABILITY_NAMED_IAM \
  --region us-east-1 \
  --tags projectname=datahub environment={ENV}

print('✅ RDS Stack deployment complete')

🗃️ Deploying RDS Stack... (This may take 10-15 minutes)

Waiting for changeset to be created..
Waiting for stack create/update to complete
Successfully created/updated stack - DataHub-RDS-dev
✅ RDS Stack deployment complete


## 9. Deploy Route53 Stack

*Creates DNS records and routing configurations*

*NOTE: This will be organization dependent, and you will need to change DNS CNAMES, A records, etc. Consult with your hostmaster for your DNS records, and whether this stack is necessary.*

In [17]:
print('🌐 Deploying Route53 Stack...')

!aws cloudformation deploy \
  --stack-name DataHub-Route53-{ENV} \
  --template-file modules/Route53.yaml \
  --parameter-overrides file://parameters-{ENV}.json \
  --capabilities CAPABILITY_NAMED_IAM \
  --region us-east-1 \
  --tags projectname=datahub environment={ENV}

print('✅ Route53 Stack deployment complete')

🌐 Deploying Route53 Stack...

Waiting for changeset to be created..
Waiting for stack create/update to complete
Successfully created/updated stack - DataHub-Route53-dev
✅ Route53 Stack deployment complete


## 10. Deploy CloudWatch Stack
*Creates monitoring, logging, and alerting infrastructure*

In [38]:
print('📊 Deploying CloudWatch Stack...')

!aws cloudformation deploy \
  --stack-name DataHub-CloudWatch-{ENV} \
  --template-file modules/CloudWatch.yaml \
  --parameter-overrides file://parameters-{ENV}.json \
  --capabilities CAPABILITY_NAMED_IAM \
  --region us-east-1 \
  --tags projectname=datahub environment={ENV}

print('✅ CloudWatch Stack deployment complete')

📊 Deploying CloudWatch Stack...

Waiting for changeset to be created..
Waiting for stack create/update to complete
Successfully created/updated stack - DataHub-CloudWatch-dev
✅ CloudWatch Stack deployment complete


## 11. Deploy SQS Stack
*Creates SQS queues for message processing*

In [23]:
print('📬 Deploying SQS Stack...')

!aws cloudformation deploy \
  --stack-name DataHub-SQS-{ENV} \
  --template-file modules/SQS.yaml \
  --parameter-overrides file://parameters-{ENV}.json \
  --capabilities CAPABILITY_NAMED_IAM \
  --region us-east-1 \
  --tags projectname=datahub environment={ENV}

print('✅ SQS Stack deployment complete')

📬 Deploying SQS Stack...

Waiting for changeset to be created..
Waiting for stack create/update to complete
Successfully created/updated stack - DataHub-SQS-dev
✅ SQS Stack deployment complete


## 12. Deploy ECR Stack
*Creates ECR repositories for container images*

In [51]:
print('📦 Deploying ECR Stack...')

!aws cloudformation deploy \
  --stack-name DataHub-ECR-{ENV} \
  --template-file modules/ECR.yaml \
  --parameter-overrides file://parameters-{ENV}.json \
  --capabilities CAPABILITY_NAMED_IAM \
  --region us-east-1 \
  --tags projectname=datahub environment={ENV}

print('✅ ECR Stack deployment complete')

📦 Deploying ECR Stack...

Waiting for changeset to be created..

No changes to deploy. Stack DataHub-ECR-dev is up to date
✅ ECR Stack deployment complete


## 13. Deploy OpenSearch Stack
*Creates the OpenSearch domain and supporting resources*

**Notice**: Please uncomment line 45-49 to create this service linked role on the first run for any new AWS Account

In [ ]:
print('🔎 Deploying OpenSearch Stack... (This may take 10-20 minutes)')

!aws cloudformation deploy \
  --stack-name DataHub-OpenSearch-{ENV} \
  --template-file modules/OpenSearch.yaml \
  --parameter-overrides file://parameters-{ENV}.json \
  --capabilities CAPABILITY_NAMED_IAM \
  --region us-east-1 \
  --tags projectname=datahub environment={ENV}


print('✅ OpenSearch Stack deployment complete')

🔎 Deploying OpenSearch Stack... (This may take 10-20 minutes)

Waiting for changeset to be created..
Waiting for stack create/update to complete
Successfully created/updated stack - DataHub-OpenSearch-dev
✅ OpenSearch Stack deployment complete


## 14. Deploy Secrets Manager Stack
*Creates secrets for database credentials, API keys, and OpenSearch configuration*

**⚠️ Important**: This stack must be deployed AFTER the OpenSearch stack because it imports the OpenSearch endpoint.

In [27]:
print('🔐 Deploying Secrets Manager Stack...')

!aws cloudformation deploy \
  --stack-name DataHub-SecretsManager-{ENV} \
  --template-file modules/SecretsManager.yaml \
  --parameter-overrides file://parameters-{ENV}.json \
  --capabilities CAPABILITY_NAMED_IAM \
  --region us-east-1 \
  --tags projectname=datahub environment={ENV}

print('✅ Secrets Manager Stack deployment complete')

🔐 Deploying Secrets Manager Stack...

Waiting for changeset to be created..
Waiting for stack create/update to complete
Successfully created/updated stack - DataHub-SecretsManager-dev
✅ Secrets Manager Stack deployment complete


## 15. Deploy ECS Stack
*Creates ECS cluster, services, and container definitions*

In [28]:
print('🐳 Deploying ECS Stack... (This may take 10-20 minutes)')

!aws cloudformation deploy \
  --stack-name DataHub-ECS-{ENV} \
  --template-file modules/ECS.yaml \
  --parameter-overrides file://parameters-{ENV}.json \
  --capabilities CAPABILITY_NAMED_IAM \
  --region us-east-1 \
  --tags projectname=datahub environment={ENV}

print('✅ ECS Stack deployment complete')

🐳 Deploying ECS Stack... (This may take 10-20 minutes)

Waiting for changeset to be created..
Waiting for stack create/update to complete
Successfully created/updated stack - DataHub-ECS-dev
✅ ECS Stack deployment complete


## 16. Deploy SES Stack
*Creates SES email identities for sending emails*

- **Important**: Email identities are region-specific, so if you change regions later, you'll need to recreate them

**Current Email Identities Created:**
- `datahub@stanford.edu` 
- `datahub.dev@stanford.edu` 
- `stanford.edu` 

### Adding Team Email Identities Outside Stanford Domain
If you need to send emails from addresses **other than @stanford.edu**, you must update this CloudFormation template.


In [19]:
print('📧 Deploying SES Stack...')

!aws cloudformation deploy \
  --stack-name DataHub-SES-{ENV} \
  --template-file modules/SES.yaml \
  --parameter-overrides file://parameters-{ENV}.json \
  --region us-east-1 \
  --tags projectname=datahub environment={ENV}

print('✅ SES Stack deployment complete')
print('')
print('📋 Next Steps:')
print('1. Go to AWS Console → SES → Verified identities (us-east-1 region)')
print('2. Verify email addresses by clicking verification links')
print('3. For production, request production access from AWS')


📧 Deploying SES Stack...
⚠️  Note: SES must be deployed in us-west-2 region

Waiting for changeset to be created..
Waiting for stack create/update to complete
Successfully created/updated stack - DataHub-SES-dev
✅ SES Stack deployment complete

📋 Next Steps:
1. Go to AWS Console → SES → Verified identities (us-east-1 region)
2. Verify email addresses by clicking verification links
3. For production, request production access from AWS


## 17. Deploy Lambda Stack
*Creates Lambda functions for data processing, automation, and integrations*

In [29]:
print('⚡ Deploying Lambda Stack...')

!aws cloudformation deploy \
  --stack-name DataHub-Lambda-{ENV} \
  --template-file modules/Lambda.yaml \
  --parameter-overrides file://parameters-{ENV}.json \
  --capabilities CAPABILITY_NAMED_IAM \
  --region us-east-1 \
  --tags projectname=datahub environment={ENV}

print('✅ Lambda Stack deployment complete')

⚡ Deploying Lambda Stack...

Waiting for changeset to be created..

No changes to deploy. Stack DataHub-Lambda-dev is up to date
✅ Lambda Stack deployment complete


In [27]:
!aws lambda update-function-code \
  --function-name DataHub-EmailService \
  --s3-bucket datahub-lambda-artifacts-stanford-dev \
  --s3-key email-service/datahub-service-email-0.0.1-SNAPSHOT-aws.jar \
  --region us-east-1

{
    "FunctionName": "DataHub-EmailService",
    "FunctionArn": "arn:aws:lambda:us-east-1:772554522152:function:DataHub-EmailService",
    "Runtime": "java17",
    "Role": "arn:aws:iam::772554522152:role/ses-sqs-lambda-role-cme1rbbm",
    "Handler": "org.springframework.cloud.function.adapter.aws.FunctionInvoker::handleRequest",
    "CodeSize": 41938550,
    "Description": "DataHub Email Service Lambda Function",
    "Timeout": 60,
    "MemorySize": 2048,
    "LastModified": "2025-12-01T18:40:28.000+0000",
    "CodeSha256": "ggC8jvJ8bwCOEpjvgHvDQJeHSvhThwpnE35VmDrv/4A=",
    "Version": "$LATEST",
    "VpcConfig": {
        "SubnetIds": [
            "subnet-02a20fa5bc848ce1a",
            "subnet-0cdbca4b512337c34"
        ],
        "SecurityGroupIds": [
            "sg-09eb41ca386a80f2a"
        ],
        "VpcId": "vpc-0fbe345a66dea8bbb",
        "Ipv6AllowedForDualStack": false
    },
    "Environment": {
        "Variables": {
            "SPRING_PROFILES_ACTIVE": "dev"
        }

## 18. Deploy EventBridge Stack *(pending)*
*Creates EventBridge rules to trigger Lambda functions on schedules*

In [22]:
print('⏰ Deploying EventBridge Stack...')

!aws cloudformation deploy \
  --stack-name DataHub-EventBridge-{ENV} \
  --template-file modules/EventBridge.yaml \
  --parameter-overrides file://parameters-{ENV}.json \
  --capabilities CAPABILITY_NAMED_IAM \
  --region us-east-1 \
  --tags projectname=datahub environment={ENV}

print('✅ EventBridge Stack deployment complete')
print('🎉 All stacks deployed! Your Data Hub is ready.')

⏰ Deploying EventBridge Stack...

Waiting for changeset to be created..
Waiting for stack create/update to complete

Failed to create/update the stack. Run the following command
to fetch the list of events leading up to the failure
aws cloudformation describe-stack-events --stack-name DataHub-EventBridge-dev
✅ EventBridge Stack deployment complete
🎉 All stacks deployed! Your Data Hub is ready.


---
# ✅ Post-Deployment Verification

After all stacks are deployed, verify the installation:

In [ ]:
print('🔍 Checking stack status...')

!aws cloudformation list-stacks \
  --stack-status-filter CREATE_COMPLETE UPDATE_COMPLETE \
  --query 'StackSummaries[?contains(StackName, `DataHub`)].{Name:StackName,Status:StackStatus}' \
  --output table

print('✅ Stack status check complete')

# 🔧 Troubleshooting

## Common Issues:

### Stack Creation Failed
```python
# Check stack events for errors
!aws cloudformation describe-stack-events --stack-name DataHub-{ENV}-[STACK-NAME]
```

### Resource Already Exists
- S3 bucket names must be globally unique
- Modify bucket names in the S3.yaml template

### Permission Denied
- Ensure your IAM user has all required permissions
- Check AWS credentials are correctly configured

### Import Value Not Found
- Ensure previous stacks completed successfully
- Verify stack names match exactly

---
# 🧹 Cleanup

**⚠️ WARNING**: This will delete ALL resources and data. Make sure you have backups of any important data.

To delete all stacks when done testing:

## Check Current Stacks Before Cleanup

In [ ]:
print('📋 Current DataHub stacks:')

!aws cloudformation list-stacks \
  --stack-status-filter CREATE_COMPLETE UPDATE_COMPLETE \
  --query 'StackSummaries[?contains(StackName, `DataHub`)].{Name:StackName,Status:StackStatus,Created:CreationTime}' \
  --output table

## Manual S3 Bucket Cleanup (Required First)
*S3 buckets with content cannot be deleted by CloudFormation*

In [ ]:
print('🗁️ Emptying S3 buckets before stack deletion...')

# List and empty all DataHub S3 buckets
import subprocess

result = subprocess.run(['aws', 's3', 'ls'], capture_output=True, text=True)
buckets = [line.split()[-1] for line in result.stdout.strip().split('\n') if 'datahub' in line.lower()]

for bucket in buckets:
    print(f'Emptying bucket: {bucket}')
    !aws s3 rm s3://{bucket} --recursive
    !aws s3api delete-bucket-versioning --bucket {bucket} --versioning-configuration Status=Suspended

print('✅ S3 buckets emptied')

## Delete CloudFormation Stacks
*Stacks must be deleted in reverse order due to dependencies*

In [ ]:
print('🗂️ Deleting stacks in reverse dependency order...')

# Delete stacks in reverse order (stack_name, region)
stacks = [
    (f'DataHub-EventBridge-{ENV}', 'us-east-1'),
    (f'DataHub-Lambda-{ENV}', 'us-east-1'),
    (f'DataHub-SES-{ENV}', 'us-east-1'),  
    (f'DataHub-ECS-{ENV}', 'us-east-1'),
    (f'DataHub-OpenSearch-{ENV}', 'us-east-1'),
    (f'DataHub-ECR-{ENV}', 'us-east-1'),
    (f'DataHub-SQS-{ENV}', 'us-east-1'),
    (f'DataHub-CloudWatch-{ENV}', 'us-east-1'),
    (f'DataHub-Route53-{ENV}', 'us-east-1'),
    (f'DataHub-RDS-{ENV}', 'us-east-1'),
    (f'DataHub-ApplicationLoadBalancer-{ENV}', 'us-east-1'),
    (f'DataHub-SecretsManager-{ENV}', 'us-east-1'),
    (f'DataHub-S3-{ENV}', 'us-east-1'),
    (f'DataHub-Networking-{ENV}', 'us-east-1')
]

for stack_name, region in stacks:
    print(f'🗁️ Deleting stack: {stack_name} (region: {region})')
    !aws cloudformation delete-stack --stack-name {stack_name} --region {region}

    print(f'⏳ Waiting for {stack_name} deletion to complete...')
    !aws cloudformation wait stack-delete-complete --stack-name {stack_name} --region {region}

    print(f'✅ {stack_name} deleted successfully\n')

print('🎉 Cleanup complete!')

## Verify Cleanup

In [ ]:
print('🔍 Checking for remaining DataHub resources...')

print('CloudFormation Stacks:')
!aws cloudformation list-stacks \
  --query 'StackSummaries[?contains(StackName, `DataHub`) && StackStatus != `DELETE_COMPLETE`].{Name:StackName,Status:StackStatus}' \
  --output table

print('\nS3 Buckets:')
result = subprocess.run(['aws', 's3', 'ls'], capture_output=True, text=True)
datahub_buckets = [line for line in result.stdout.split('\n') if 'datahub' in line.lower()]
if datahub_buckets:
    for bucket in datahub_buckets:
        print(bucket)
else:
    print('No DataHub S3 buckets found')

print('\nLoad Balancers:')
!aws elbv2 describe-load-balancers \
  --query 'LoadBalancers[?contains(LoadBalancerName, `DataHub`)].LoadBalancerName' \
  --output text || echo 'No DataHub load balancers found'

---
# 📚 Additional Information

## Architecture Overview
- **VPC**: Isolated network environment
- **Public Subnets**: ALB and NAT Gateway
- **Private Subnets**: ECS services and RDS
- **Security Groups**: Network access control
- **ECS Fargate**: Serverless container hosting
- **RDS**: Managed database service
- **S3**: Object storage for files and artifacts

## Cost Optimization Tips
- Use `t3.micro` or `t3.small` for development
- Enable S3 lifecycle policies
- Set up CloudWatch billing alerts
- Delete unused resources regularly

## Security Considerations
- Rotate AWS credentials regularly
- Use least privilege IAM policies
- Enable VPC Flow Logs
- Monitor CloudTrail logs

## Support
For issues or questions:
1. Check CloudFormation events in AWS Console
2. Review CloudWatch logs
3. Consult AWS documentation
4. Contact your AWS support team

---
# 📚 Next Steps


1. **Deploy Containers to ECR**: ...
2. **Obtain Domains and SSL Certificates**: ...
3. **Replace Lambda code with your actual code via S3 or container images.**
